# 01 — Kiểm toán và làm sạch dữ liệu Airbnb Bangkok

**Nhóm 10 — IE313.F32.LT.CNTT.** Nguồn: Inside Airbnb, snapshot phát hành 29/06/2026; hạn nộp 18/10/2026.

Notebook thực thi Bước 1 và Bước 3–4 theo `docs/SPECIFICATION.md` và `docs/CLEANING_PLAN.md`. Mục tiêu: truy nguồn, xác minh cấu trúc/giá, tạo baseline và nhật ký trước/sau. Kết quả thống kê suy luận và mô hình được thực hiện ở notebook sau.

**Bài học:** TH01/Bài01 (Pandas, I/O), Bài02 (missing, ép kiểu). SHA-256, kiểm thử, bảo toàn raw và tách nhánh chống rò rỉ là quy chuẩn kỹ thuật repository. Minh họa scaling/binning/dummy được dành cho nhánh EDA riêng, không fit trên baseline dùng chung.

In [1]:
import json
import platform
import sys
from pathlib import Path

import pandas as pd
from IPython.display import display

candidates = [Path.cwd(), Path.cwd().parent, Path.cwd() / "workspaces/final-project"]
PROJECT = next(
    (
        p
        for p in candidates
        if (p / "data/source_manifest.json").is_file() and (p / "src/wrangling.py").is_file()
    ),
    None,
)
if PROJECT is None:
    raise RuntimeError("Chạy từ gốc đồ án hoặc thư mục notebooks.")
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))
from src.data_audit import run_full_audit  # noqa: E402
from src.data_loader import compute_sha256  # noqa: E402
from src.wrangling import MODEL_FEATURES, run_cleaning  # noqa: E402

print({"python": platform.python_version(), "pandas": pd.__version__})

{'python': '3.12.3', 'pandas': '3.0.6'}


## 1. Nguồn gốc và toàn vẹn

File nguồn giữ nguyên. Manifest ghi URL nguồn, giấy phép CC BY 4.0 và SHA-256. Ký hiệu `$` là định dạng xuất, đơn vị THB được đối chiếu với từ điển và ký hiệu ฿ trong quote; không quy đổi tiền tệ. Đối chiếu giá/đêm chỉ áp dụng cho các cặp dữ liệu hữu hạn so sánh được.

In [2]:
manifest = json.loads((PROJECT / "data/source_manifest.json").read_text())
raw = PROJECT / manifest["relative_path"]
raw_before = (compute_sha256(raw), raw.stat().st_mtime_ns)
assert raw_before[0] == manifest["sha256"]
audit = run_full_audit(
    raw,
    PROJECT / "reports/audit",
    manifest["sha256"],
    markdown_output_path=PROJECT / "docs/DATA_AUDIT.md",
)
display(pd.DataFrame([audit["structure"]]).drop(columns="dtype_counts"))
display(
    pd.DataFrame([audit["currency_and_granularity"].get("price_vs_quote_per_night_comparison", {})])
)

[*] Đang tính toán mã băm SHA-256 cho: bangkok_listings_2026-06-29.csv.gz...
[*] Đang nạp dữ liệu thô vào bộ nhớ...


[*] Đang kiểm toán cấu trúc dữ liệu và tính đơn nhất...
[*] Đang kiểm toán phạm vi thời gian cào dữ liệu...
[*] Đang kiểm toán biến giá mục tiêu (price)...


[*] Đang kiểm toán bằng chứng tiền tệ và kỳ lưu trú...
[*] Đang kiểm toán 12 biến ứng viên...
[*] Đang kiểm toán phân bố chủ nhà (host clustering)...
[✓] Đã xuất file JSON kiểm toán tổng hợp: /var/www/lienthongdaihoc/IE313.F32.LT.CNTT.PhanTichVaTrucQuanDuLieu/workspaces/final-project/reports/audit/bangkok_audit_summary.json


[✓] Đã xuất file CSV danh mục 90 cột: /var/www/lienthongdaihoc/IE313.F32.LT.CNTT.PhanTichVaTrucQuanDuLieu/workspaces/final-project/reports/audit/bangkok_columns.csv
[✓] Đã đồng bộ tài liệu Markdown kiểm toán: /var/www/lienthongdaihoc/IE313.F32.LT.CNTT.PhanTichVaTrucQuanDuLieu/workspaces/final-project/docs/DATA_AUDIT.md


,total_rows,total_columns,total_cells,total_missing_cells,total_missing_cells_pct,complete_duplicate_rows,has_id_column,unique_ids_count,duplicate_id_rows,is_one_row_per_listing
0,31069,90,2796210,581427,20.79,0,True,31069,0,True


,total_compared,exact_matches,exact_match_rate_pct,tolerance_threshold,tolerance_matches,tolerance_match_rate_pct,max_difference,mean_difference
0,28987,28987,100.0,0.05,28987,100.0,0.0,0.0


## 2. Chính sách làm sạch

Loại dòng giá thiếu, sai định dạng, không hữu hạn hoặc không dương; không điền giá. Chỉ loại dòng trùng toàn bộ; ID thiếu/trùng khác nội dung thì dừng. Chuẩn hóa kiểu số/danh mục theo quy tắc định trước, giữ missing predictor. Phòng ngủ 0 vẫn giữ, `bathrooms_text` chưa dùng tự điền vì chưa kiểm chứng quy tắc.

Giá dương rất nhỏ/lớn vẫn giữ. IQR chỉ tạo danh sách rà soát, không lọc baseline. Không dùng ngưỡng 100 THB, không winsorize/log. Xem bảng **Vấn đề–Xử lý–Lý do–Ảnh hưởng** đầy đủ tại `docs/CLEANING_PLAN.md`.

In [3]:
cleaning = run_cleaning(
    raw,
    PROJECT / "data/source_manifest.json",
    PROJECT / "data/processed",
    PROJECT / "reports/cleaning",
)
clean = pd.read_csv(PROJECT / "data/processed/bangkok_cleaned.csv")
excluded = pd.read_csv(PROJECT / "reports/cleaning/excluded_rows.csv")
ledger = pd.DataFrame(
    [
        {"Giai đoạn": "Raw", "Số dòng": cleaning["rows_before"]},
        {
            "Giai đoạn": "Sau loại trùng toàn bộ",
            "Số dòng": cleaning["rows_before"] - cleaning["exact_duplicates_removed"],
        },
        {"Giai đoạn": "Sau kiểm tra giá / baseline", "Số dòng": cleaning["rows_after"]},
    ]
)
display(ledger)
display(excluded.groupby("reason").size().rename("Số dòng loại").to_frame())
assert len(clean) + len(excluded) == cleaning["rows_before"]
assert clean["id"].notna().all() and clean["id"].is_unique
assert clean["price"].notna().all() and (clean["price"] > 0).all()
assert (
    compute_sha256(PROJECT / "data/processed/bangkok_cleaned.csv") == cleaning["processed_sha256"]
)

,Giai đoạn,Số dòng
0,Raw,31069
1,Sau loại trùng toàn bộ,31069
2,Sau kiểm tra giá / baseline,28987


,Số dòng loại
reason,
missing_price,2082


## 3. Khuyết còn lại và ngoại lệ

Bảng sau mô tả **mẫu sau khi loại dòng thiếu giá**, nên số khuyết predictor có thể khác bảng raw. EDA phải ghi n theo từng cặp/nhóm; không tự drop mọi dòng thiếu predictor. Ngoại lệ giá chưa được xác định là lỗi hay listing cao cấp.

In [4]:
missing = pd.DataFrame({"Số khuyết": clean[MODEL_FEATURES].isna().sum()})
missing["Tỷ lệ (%)"] = (missing["Số khuyết"] / len(clean) * 100).round(2)
display(missing)
display(pd.DataFrame([cleaning["outlier_review"]]))
outliers = pd.read_csv(PROJECT / "reports/cleaning/price_outliers.csv")
display(outliers.sort_values("price", ascending=False).head(5))
assert cleaning["outlier_review"]["removed_count"] == 0
assert clean["price"].min() == cleaning["price_min_retained"]
assert clean["price"].max() == cleaning["price_max_retained"]

,Số khuyết,Tỷ lệ (%)
room_type,0,0.00
property_type,0,0.00
neighbourhood_cleansed,0,0.00
accommodates,0,0.00
bedrooms,3559,12.28
bathrooms,2701,9.32
beds,1270,4.38
minimum_nights,2,0.01
number_of_reviews,0,0.00
review_scores_rating,9451,32.60


,q1,q3,lower_bound,upper_bound,flagged_count,removed_count,use
0,1041.465,2539.565,-1205.685,4786.715,2730,0,EDA_ONLY


,id,price,room_type,neighbourhood_cleansed,review_reason
411,21462529,1320255.00,Private room,Ratchathewi,outside_1_5_iqr_not_automatically_invalid
2348,15380964,1159508.00,Entire home/apt,Huai Khwang,outside_1_5_iqr_not_automatically_invalid
2713,26511695,1157979.00,Entire home/apt,Parthum Wan,outside_1_5_iqr_not_automatically_invalid
203,23740993,1157401.00,Entire home/apt,Huai Khwang,outside_1_5_iqr_not_automatically_invalid
124,1376399745235368707,1118917.14,Entire home/apt,Khlong Toei,outside_1_5_iqr_not_automatically_invalid


## 4. Bàn giao hai nhánh và giới hạn

Baseline chưa điền khuyết, chuẩn hóa, mã hóa hoặc chọn biến theo dữ liệu. Nhánh EDA đọc các biến cần thiết và ghi n. Nhánh mô hình sau này lấy X theo allowlist, y = `price`, chia 70/30 seed 42; mọi phép học dữ liệu nằm trong training fold/Pipeline và CV 4-fold. Không truyền cờ IQR toàn bộ dữ liệu, `price_raw`, ID/URL/quote/doanh thu vào X.

Một listing/snapshot không có nghĩa độc lập thống kê: nhiều listing cùng host; p-value thông thường chỉ mang tính thăm dò. Chia theo listing không chứng minh tổng quát hóa sang chủ nhà mới. Dữ liệu giá niêm yết có thể khác giá giao dịch, và việc loại thiếu giá có thể gây sai lệch chọn mẫu.

In [5]:
assert set(MODEL_FEATURES).issubset(clean.columns)
assert not set(MODEL_FEATURES) & {"price", "price_raw", "id", "host_id", "last_scraped"}
assert not any(name.startswith("price_quote_") for name in MODEL_FEATURES)
assert not cleaning["imputation_applied"] and not cleaning["scaling_applied"]
assert not cleaning["feature_selection_applied"]
assert raw_before == (compute_sha256(raw), raw.stat().st_mtime_ns)
print("Kiểm tra bàn giao đạt: baseline, nhật ký và tính bất biến raw.")
print("Bước tiếp theo: notebook 02 — EDA giá và kiểm tra điều kiện Pearson/ANOVA.")

Kiểm tra bàn giao đạt: baseline, nhật ký và tính bất biến raw.
Bước tiếp theo: notebook 02 — EDA giá và kiểm tra điều kiện Pearson/ANOVA.


## Tài liệu và trách nhiệm

- `docs/DATA_AUDIT.md`, `docs/CLEANING_PLAN.md`, `docs/RUNBOOK.md`.
- Inside Airbnb: https://insideairbnb.com/get-the-data/ ; nguồn/từ điển cụ thể trong manifest.
- Phụ trách chặng: Lê Minh Thiện; kiểm tra chéo: Phạm Quốc Thắng. Việc chạy tự động không thay thế phần giải thích và kiểm tra của hai thành viên.
